# 06 — Aula 6: validación, excepciones y filtrado

## Objetivo
Practicar manejo de errores en Python y filtrar reseñas negativas del desafío anterior.

## Conceptos
- **`try/except`**: capturar errores (`ValueError`, `TypeError`)
- **Filtrado booleano en Pandas**: `df[df["col"] == valor]`
- **`join()`**: unir textos con un separador
- **Categorización con LLM**: pedir 5 categorías de reclamos a Groq

## Dependencia
Ejecutar antes el notebook **05** para tener `df_reviews` con la columna `Sentimiento`.


In [1]:
import sys
sys.path.insert(0, "..")

from notebooks._setup import load_env, require_key, DATOS_DIR, OUTPUT_DIR, safe_generate, use_real_apis, LocalGeminiClient, LocalGroqClient

load_env()
OUTPUT_DIR.mkdir(exist_ok=True)

require_key("GROQ_API_KEY")
import pandas as pd

if use_real_apis():
    from groq import Groq
    client_groq = Groq()
else:
    client_groq = LocalGroqClient()
reviews_path = OUTPUT_DIR / "reviews_sentimiento.csv"
if reviews_path.exists():
    df_reviews = pd.read_csv(reviews_path)
else:
    df_reviews = pd.read_csv(DATOS_DIR / "reviews.csv")
    df_reviews["Sentimiento"] = ["Negativa" if i % 3 == 0 else "Positiva" if i % 3 == 1 else "Neutra" for i in range(len(df_reviews))]
    df_reviews.to_csv(reviews_path, index=False)

df_reviews.head()

,reviewerID,asin,reviewerName,reviewText,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote,Sentimiento
0,A2FII3I2MBMUIA,B007WTAJTO,1K3,it works as expected. I should have sprung for...,1356220800,23/12/2012,715,0,0,Neutra
1,A3H99DFEG68SR,B007WTAJTO,1m2,This think has worked out great.Had a diff. br...,1384992000,21/11/2013,382,0,0,Positiva
2,A375ZM4U047O79,B007WTAJTO,2&amp;1/2Men,"Bought it with Retail Packaging, arrived legit...",1373673600,13/07/2013,513,0,0,Neutra
3,A2IDCSC6NVONIZ,B007WTAJTO,2Cents!,It's mini storage. It doesn't do anything els...,1367193600,29/04/2013,588,0,0,Neutra
4,A26YHXZD5UFPVQ,B007WTAJTO,2K1Toaster,I have it in my phone and it never skips a bea...,1382140800,19/10/2013,415,0,0,Positiva


# Aula 6


In [2]:
edad = 25
print(f"Edad de ejemplo: {edad}")

Edad de ejemplo: 25


In [3]:
try:
    persona = 6
    persona2 = "Juan, Pedro, Laura, Maria"
    persona + persona2
except TypeError as te:
    print("No puedes sumar un numero con un texto")
    print("El error es:", te)

No puedes sumar un numero con un texto
El error es: unsupported operand type(s) for +: 'int' and 'str'


In [4]:
try:
    edad = int("25")
except ValueError:
    print("No ingresaste un numero")
else:
    print(f"Edad valida: {edad}")

Edad valida: 25


In [5]:
try:
    edad = int("texto")
except ValueError as ve:
    print("No ingresaste un numero")
    print("El error es:", ve)

No ingresaste un numero
El error es: invalid literal for int() with base 10: 'texto'


In [6]:
try:
    persona = 6
    persona2 = "Juan, Pedro, Laura, Maria"
    persona + persona2
except ValueError as ve:
    print("No ingresaste un numero")
    print("El error es:", ve)
except TypeError as te:
    print("No puedes sumar un numero con un texto")
    print("El error es:", te)

No puedes sumar un numero con un texto
El error es: unsupported operand type(s) for +: 'int' and 'str'


In [7]:
df_reviews
df_reviews["Sentimiento"]

0       Neutra
1     Positiva
2       Neutra
3       Neutra
4     Positiva
5       Neutra
6       Neutra
7       Neutra
8       Neutra
9       Neutra
10      Neutra
11      Neutra
12    Positiva
13      Neutra
14      Neutra
15      Neutra
16      Neutra
17    Positiva
18      Neutra
19      Neutra
20      Neutra
21      Neutra
22      Neutra
23      Neutra
24    Positiva
25      Neutra
26      Neutra
27      Neutra
28      Neutra
29    Negativa
30      Neutra
31      Neutra
32    Positiva
33      Neutra
34      Neutra
35    Positiva
36      Neutra
37      Neutra
38      Neutra
39      Neutra
Name: Sentimiento, dtype: str

In [8]:
df_reviews["Sentimiento"] == "Negativa"

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29     True
30    False
31    False
32    False
33    False
34    False
35    False
36    False
37    False
38    False
39    False
Name: Sentimiento, dtype: bool

In [9]:
df_reviews_negativas = df_reviews[df_reviews["Sentimiento"] == "Negativa"]

In [10]:
df_reviews_negativas

,reviewerID,asin,reviewerName,reviewText,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote,Sentimiento
29,A1FZKP9XTKMALD,B007WTAJTO,Amazon Customer,"Does what it says. Hasn't broken yet, and I ho...",1375401600,08/02/2013,668,0,1,Negativa


In [11]:
reviews_negativas =df_reviews_negativas["reviewText"]
reviews_negativas

29    Does what it says. Hasn't broken yet, and I ho...
Name: reviewText, dtype: str

In [12]:
reviews_negativas_unidas= "####".join(reviews_negativas)
reviews_negativas_unidas

"Does what it says. Hasn't broken yet, and I hope it doesn't break. I've had luck with this brand before. Usually get bored with the product before it breaks."

In [13]:
prompt = f"""
Eres un analista de datos. Recibiras resenas negativas separadas por ####.
Devuelve categorias principales separadas por coma.

{reviews_negativas_unidas}
"""
categorias = safe_generate(
    lambda texto: client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": texto}],
    ).choices[0].message.content,
    prompt,
    task="categorias",
)
print(categorias)

calidad del producto, entrega, precio, atencion al cliente
